# ERGT Colab Phase 3 Alpha Sweep

Run this notebook on Google Colab with `Runtime > Change runtime type > GPU`.

This diagnostic sweep tests whether the short proof result is alpha-sensitive. It runs four new conditions: real_d/random_d at alpha `0.05` and `0.2`. The existing short proof alpha `0.1` runs are reused from the repository.

In [ ]:
REPO_URL = "https://github.com/jalaljafari2009/ERGT.git"
BRANCH = "main"
PROJECT_DIR = "/content/ERGT"

DATA_DIR = "data/processed/wikitext2_gpt2_ctx256"
DEVICE = "cuda"
BASELINE_RESULTS = "runs/phase0_baseline/short_proof_wikitext2/baseline_results.json"
OUTPUT_DIR = "runs/phase3_geo_attention/alpha_sweep_short_proof"

RUN_SWEEP = True
RUN_COMPARISON = True
FORCE_RERUN = False

SWEEP_CONFIGS = {
    "real_d_alpha_0_05": (
        "configs/ergt_v1/alpha_sweep/short_proof_real_d_alpha_0_05.json"
    ),
    "real_d_alpha_0_2": (
        "configs/ergt_v1/alpha_sweep/short_proof_real_d_alpha_0_2.json"
    ),
    "random_d_alpha_0_05": (
        "configs/ergt_v1/alpha_sweep/short_proof_random_d_alpha_0_05.json"
    ),
    "random_d_alpha_0_2": (
        "configs/ergt_v1/alpha_sweep/short_proof_random_d_alpha_0_2.json"
    ),
}

SWEEP_RESULTS = {
    "real_d_alpha_0_05": (
        "runs/phase3_geo_attention/alpha_sweep_short_proof/"
        "real_d_alpha_0_05/metrics.json"
    ),
    "real_d_alpha_0_1": "runs/phase3_geo_attention/short_proof_real_d/metrics.json",
    "real_d_alpha_0_2": (
        "runs/phase3_geo_attention/alpha_sweep_short_proof/"
        "real_d_alpha_0_2/metrics.json"
    ),
    "random_d_alpha_0_05": (
        "runs/phase3_geo_attention/alpha_sweep_short_proof/"
        "random_d_alpha_0_05/metrics.json"
    ),
    "random_d_alpha_0_1": "runs/phase3_geo_attention/short_proof_random_d/metrics.json",
    "random_d_alpha_0_2": (
        "runs/phase3_geo_attention/alpha_sweep_short_proof/"
        "random_d_alpha_0_2/metrics.json"
    ),
}

## 1. Runtime probe

In [ ]:
import json
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

import torch

print("cwd:", os.getcwd())
print("python:", sys.version)
print("platform:", platform.platform())
print("COLAB_RELEASE_TAG:", os.environ.get("COLAB_RELEASE_TAG"))
print("COLAB_GPU:", os.environ.get("COLAB_GPU"))

nvidia_smi = shutil.which("nvidia-smi")
print("nvidia-smi:", nvidia_smi)
if nvidia_smi:
    subprocess.run([nvidia_smi], check=False)

print("torch:", torch.__version__)
print("cuda_available:", torch.cuda.is_available())
print("cuda_device_count:", torch.cuda.device_count())
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. Set Colab hardware accelerator to GPU.")
print("cuda_device_name:", torch.cuda.get_device_name(0))

## 2. Clone/update repo and install dependencies

In [ ]:
project_path = Path(PROJECT_DIR)
if project_path.exists():
    subprocess.run(["git", "-C", PROJECT_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", PROJECT_DIR, "checkout", BRANCH], check=True)
    subprocess.run(
        ["git", "-C", PROJECT_DIR, "pull", "--ff-only", "origin", BRANCH],
        check=True,
    )
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, PROJECT_DIR], check=True)

os.chdir(PROJECT_DIR)
repo_head = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"], text=True
)
print("repo:", repo_head.strip())
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev,data,viz]"],
    check=True,
)

## 3. Prepare WikiText-2

In [ ]:
subprocess.run(
    [
        sys.executable,
        "scripts/prepare_wikitext2.py",
        "--output-dir",
        DATA_DIR,
        "--tokenizer",
        "gpt2",
        "--context-length",
        "256",
    ],
    cwd=PROJECT_DIR,
    check=True,
)

## 4. Run alpha sweep

In [ ]:
def run_command(command):
    print("\n$", " ".join(command))
    subprocess.run(command, cwd=PROJECT_DIR, check=True)


if RUN_SWEEP:
    for label, config_path in SWEEP_CONFIGS.items():
        result_path = Path(PROJECT_DIR) / SWEEP_RESULTS[label]
        if result_path.exists() and not FORCE_RERUN:
            print("Skipping existing sweep run:", label, result_path)
            continue
        print("\n=== Alpha sweep run:", label, "===")
        run_command(
            [
                sys.executable,
                "experiments/train_ergt_v1.py",
                "--config",
                config_path,
                "--data-dir",
                DATA_DIR,
                "--device",
                DEVICE,
            ]
        )
else:
    print("RUN_SWEEP is False; no sweep run executed.")

for label, path in SWEEP_RESULTS.items():
    if not (Path(PROJECT_DIR) / path).exists():
        raise FileNotFoundError(f"Missing sweep result for {label}: {path}")

## 5. Compare alpha sweep

In [ ]:
run_specs = [
    f"real_d:0.05:{SWEEP_RESULTS['real_d_alpha_0_05']}",
    f"real_d:0.1:{SWEEP_RESULTS['real_d_alpha_0_1']}",
    f"real_d:0.2:{SWEEP_RESULTS['real_d_alpha_0_2']}",
    f"random_d:0.05:{SWEEP_RESULTS['random_d_alpha_0_05']}",
    f"random_d:0.1:{SWEEP_RESULTS['random_d_alpha_0_1']}",
    f"random_d:0.2:{SWEEP_RESULTS['random_d_alpha_0_2']}",
]

if RUN_COMPARISON:
    command = [
        sys.executable,
        "experiments/compare_alpha_sweep.py",
        "--baseline",
        BASELINE_RESULTS,
        "--output-dir",
        OUTPUT_DIR,
    ]
    for spec in run_specs:
        command.extend(["--run", spec])
    run_command(command)
else:
    print("RUN_COMPARISON is False; no sweep comparison generated.")

## 6. Print alpha sweep report

In [ ]:
report_path = Path(PROJECT_DIR) / OUTPUT_DIR / "alpha_sweep_results.json"
with report_path.open("r", encoding="utf-8") as handle:
    report = json.load(handle)

print(json.dumps(report, indent=2, sort_keys=True)[:12000])

## 7. Archive light outputs

In [ ]:
light_root = Path("/content/ergt_alpha_sweep_short_proof_light")
if light_root.exists():
    shutil.rmtree(light_root)

include_roots = [
    "runs/phase3_geo_attention/alpha_sweep_short_proof",
]

for include_root in include_roots:
    source_dir = Path(PROJECT_DIR) / include_root
    target_dir = light_root / include_root
    target_dir.mkdir(parents=True, exist_ok=True)
    for path in source_dir.rglob("*"):
        if not path.is_file() or path.suffix not in {".json", ".jsonl"}:
            continue
        relative = path.relative_to(source_dir)
        destination = target_dir / relative
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, destination)
        print("included:", destination.relative_to(light_root))

light_archive = shutil.make_archive(
    "/content/ergt_alpha_sweep_short_proof_light", "zip", light_root
)
print("Light archive ready:", light_archive)